# Dataset Inventory

## Objective

The goal of this notebook is to understand the structure of the FlyRank internship warehouse before selecting a research direction.

We will inspect the available tables, their schemas, row counts, and relationships between them.

In [1]:
from huggingface_hub import hf_hub_download
import duckdb

local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_clients.parquet",
    repo_type="dataset"
)

print("Downloaded to:", local_file)

con = duckdb.connect()

client_schema = con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
""", [local_file]).fetchdf()

client_schema


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\dim_clients.parquet


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [2]:
content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

print("Downloaded to:", content_file)

Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\dim_content.parquet


In [3]:
client_stats = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(client_hash_id) AS client_id_count,
        COUNT(DISTINCT client_hash_id) AS unique_client_id_count
    FROM read_parquet(?)
""", [local_file]).fetchdf()

client_stats

,row_count,client_id_count,unique_client_id_count
0,104,104,104


In [4]:
content_stats = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(content_hash_id) AS content_id_count,
        COUNT(DISTINCT content_hash_id) AS unique_content_id_count
    FROM read_parquet(?)
""", [content_file]).fetchdf()

content_stats

,row_count,content_id_count,unique_content_id_count
0,519606,519606,519606


In [5]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

files


['.gitattributes',
 'README.md',
 'dim_clients.parquet',
 'dim_content.parquet',
 'fact_content_daily_performance/month=2025-01/data_0.parquet',
 'fact_content_daily_performance/month=2025-02/data_0.parquet',
 'fact_content_daily_performance/month=2025-03/data_0.parquet',
 'fact_content_daily_performance/month=2025-04/data_0.parquet',
 'fact_content_daily_performance/month=2025-05/data_0.parquet',
 'fact_content_daily_performance/month=2025-06/data_0.parquet',
 'fact_content_daily_performance/month=2025-07/data_0.parquet',
 'fact_content_daily_performance/month=2025-08/data_0.parquet',
 'fact_content_daily_performance/month=2025-09/data_0.parquet',
 'fact_content_daily_performance/month=2025-10/data_0.parquet',
 'fact_content_daily_performance/month=2025-11/data_0.parquet',
 'fact_content_daily_performance/month=2025-12/data_0.parquet',
 'fact_content_daily_performance/month=2026-01/data_0.parquet',
 'fact_content_daily_performance/month=2026-02/data_0.parquet',
 'fact_content_daily_pe

In [7]:

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print("Downloaded to:", performance_file)


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance_sample.parquet


In [8]:
performance_stats = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(report_date) AS date_count,
        COUNT(client_hash_id) AS client_id_count,
        COUNT(content_hash_id) AS content_id_count,
        COUNT(DISTINCT report_date) AS unique_dates,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_contents
    FROM read_parquet(?)
""", [performance_file]).fetchdf()

performance_stats

,row_count,date_count,client_id_count,content_id_count,unique_dates,unique_clients,unique_contents
0,11694072,11694072,11694072,11694072,30,65,409205


In [9]:
performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print("Downloaded to:", performance_file)


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance_sample.parquet


In [10]:
duplicate_grain = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(?)
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""", [performance_file]).fetchdf()

duplicate_grain


,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-16,client_1a8bf67cad4ee525,content_c48b57d906d93d6d,2
1,2026-06-24,client_def0955f7a377868,content_40c868cfcb49537c,2
2,2026-06-27,client_86ebc2f12c01f586,content_cd422cd81fb0a0b5,2
3,2026-06-29,client_1a730cb2640a1abf,content_8a5dd4ef03e333e8,2
4,2026-06-19,client_06d356715a8ff3b6,content_a0e5d499009cc097,2
5,2026-06-19,client_1a8bf67cad4ee525,content_14d11fe89c688c0f,2
6,2026-06-22,client_810019792c9b8efc,content_ef8338f4365f7423,2
7,2026-06-27,client_aef6ffea193da149,content_36bd0582947f18ff,2
8,2026-06-14,client_b77d0d5f08f05e64,content_65a9bd83aebd0e58,2
9,2026-06-14,client_86ebc2f12c01f586,content_0e2770bf401d2de1,2


In [11]:
duplicate_rows = con.execute("""
    SELECT *
    FROM read_parquet(?)
    WHERE report_date = '2026-06-28'
      AND client_hash_id = 'client_810019792c9b8efc'
      AND content_hash_id = 'content_0064be1867cf7ae8'
""", [performance_file]).fetchdf()

duplicate_rows

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-28,client_810019792c9b8efc,content_0064be1867cf7ae8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-28,client_810019792c9b8efc,content_0064be1867cf7ae8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [12]:
duplicate_check = con.execute("""
    SELECT COUNT(*) AS unique_rows
    FROM (
        SELECT DISTINCT *
        FROM read_parquet(?)
        WHERE report_date = '2026-06-28'
          AND client_hash_id = 'client_810019792c9b8efc'
          AND content_hash_id = 'content_0064be1867cf7ae8'
    )
""", [performance_file]).fetchdf()

duplicate_check

,unique_rows
0,1


In [13]:
duplicate_summary = con.execute("""
    SELECT
        COUNT(*) AS total_groups,
        SUM(CASE WHEN row_count > 1 THEN 1 ELSE 0 END) AS duplicate_groups,
        SUM(row_count - 1) AS duplicate_rows
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
    )
""", [performance_file]).fetchdf()

duplicate_summary

,total_groups,duplicate_groups,duplicate_rows
0,11687682,6390.0,6390.0


In [14]:
query_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset"
)

print("Downloaded to:", query_file)


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_query_90d.parquet


In [15]:
query_grain = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_contents,
        COUNT(DISTINCT query_hash_id) AS unique_queries
    FROM read_parquet(?)
""", [query_file]).fetchdf()

query_grain

,total_rows,unique_clients,unique_contents,unique_queries
0,2414248,52,133852,1180090


In [16]:
query_duplicate_check = con.execute("""
    SELECT
        COUNT(*) AS total_groups,
        SUM(CASE WHEN row_count > 1 THEN 1 ELSE 0 END) AS duplicate_groups,
        SUM(row_count - 1) AS duplicate_rows
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            query_hash_id,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        GROUP BY
            client_hash_id,
            content_hash_id,
            query_hash_id
    )
""", [query_file]).fetchdf()

query_duplicate_check


,total_groups,duplicate_groups,duplicate_rows
0,2414248,0.0,0.0


In [17]:
client_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT c.client_hash_id) AS content_clients,
        COUNT(DISTINCT d.client_hash_id) AS matched_clients
    FROM read_parquet(?) c
    LEFT JOIN read_parquet(?) d
        ON c.client_hash_id = d.client_hash_id
    WHERE d.client_hash_id IS NOT NULL
""", [content_file, local_file]).fetchdf()

client_relationship

,content_clients,matched_clients
0,84,84


In [18]:
content_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT p.content_hash_id) AS performance_contents,
        COUNT(DISTINCT c.content_hash_id) AS matched_contents
    FROM read_parquet(?) p
    LEFT JOIN read_parquet(?) c
        ON p.content_hash_id = c.content_hash_id
    WHERE c.content_hash_id IS NOT NULL
""", [performance_file, content_file]).fetchdf()

content_relationship

,performance_contents,matched_contents
0,409205,409205


In [19]:
query_content_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT q.content_hash_id) AS query_contents,
        COUNT(DISTINCT c.content_hash_id) AS matched_contents
    FROM read_parquet(?) q
    LEFT JOIN read_parquet(?) c
        ON q.content_hash_id = c.content_hash_id
    WHERE c.content_hash_id IS NOT NULL
""", [query_file, content_file]).fetchdf()

query_content_relationship

,query_contents,matched_contents
0,133852,133852


In [20]:
query_client_relationship = con.execute("""
    SELECT
        COUNT(DISTINCT q.client_hash_id) AS query_clients,
        COUNT(DISTINCT c.client_hash_id) AS matched_clients
    FROM read_parquet(?) q
    LEFT JOIN read_parquet(?) c
        ON q.client_hash_id = c.client_hash_id
    WHERE c.client_hash_id IS NOT NULL
""", [query_file, local_file]).fetchdf()

query_client_relationship

,query_clients,matched_clients
0,52,52


                    dim_clients
                         │
                   client_hash_id
                         │
                         ▼
                    dim_content
                  ┌──────┴──────┐
                  │             │
          content_hash_id   content_hash_id
                  │             │
                  ▼             ▼
        daily_performance    query_90d
          date + client      client + content
          + content           + query